# Model Evaluation & Performance Improvement
### Internship Track: Week 3 – Task 3
**Author:** Sayantani Adhikary  
**Objective:** Evaluate multiple classification models, diagnose underfitting and overfitting using train vs. test metrics, and improve performance through regularization and feature scaling.


## 1. Import Libraries & Load Dataset
We use Scikit-Learn's built-in Breast Cancer Wisconsin dataset (569 samples, 30 numerical features, binary target: malignant vs. benign).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

sns.set_theme(style="whitegrid")
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
print(f"Dataset Shape: {X.shape}")


## 2. Train-Test Split & Feature Scaling
Split data into 80% training and 20% testing sets with stratification, then apply `StandardScaler` (fitted strictly on training data to prevent data leakage).


In [1]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")


## 3. Model Comparison: Diagnosing Underfitting, Overfitting & Improvement
We compare four models:
1. **Decision Tree (`max_depth=1`)**: Demonstrates **underfitting** (too simple to capture patterns).
2. **Decision Tree (`max_depth=None`)**: Demonstrates **overfitting** (100% train accuracy, lower test generalization).
3. **Regularized Random Forest (`max_depth=5`)**: Reduces variance through ensemble bagging and depth limits.
4. **Logistic Regression (Scaled)**: Linear baseline with L2 regularization achieving optimal generalization.


In [1]:
models = {
    'Decision Tree (Depth=1 / Underfit)': (DecisionTreeClassifier(max_depth=1, random_state=42), X_train, X_test),
    'Decision Tree (Unconstrained / Overfit)': (DecisionTreeClassifier(max_depth=None, random_state=42), X_train, X_test),
    'Random Forest (Regularized Depth=5)': (RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), X_train, X_test),
    'Logistic Regression (Scaled + L2)': (LogisticRegression(max_iter=1000, random_state=42), X_train_scaled, X_test_scaled)
}

results = []
for name, (model, X_tr, X_te) in models.items():
    model.fit(X_tr, y_train)
    train_preds = model.predict(X_tr)
    test_preds = model.predict(X_te)
    test_probs = model.predict_proba(X_te)[:, 1]

    results.append({
        'Model': name,
        'Train Acc (%)': round(accuracy_score(y_train, train_preds) * 100, 2),
        'Test Acc (%)': round(accuracy_score(y_test, test_preds) * 100, 2),
        'Precision (%)': round(precision_score(y_test, test_preds) * 100, 2),
        'Recall (%)': round(recall_score(y_test, test_preds) * 100, 2),
        'F1-Score': round(f1_score(y_test, test_preds), 4),
        'ROC-AUC': round(roc_auc_score(y_test, test_probs), 4)
    })

comparison_df = pd.DataFrame(results)
print("=== Performance Comparison Table ===")
print(comparison_df.to_string(index=False))


## 4. Visual Comparison: Overfitting vs. Generalization
Plot Training vs. Testing accuracy to highlight the gap in unconstrained trees versus regularized models.


In [1]:
melted_df = comparison_df.melt(id_vars='Model', value_vars=['Train Acc (%)', 'Test Acc (%)'], var_name='Split', value_name='Accuracy')

plt.figure(figsize=(10, 5))
sns.barplot(x='Model', y='Accuracy', hue='Split', data=melted_df, palette='Set2')
plt.title('Train vs. Test Accuracy: Diagnosing Underfitting & Overfitting', fontsize=12, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.ylim(80, 102)
plt.tight_layout()
plt.show()


## 5. Best Model Confusion Matrix (Logistic Regression)
Inspect false positives and false negatives for the top-performing model.


In [1]:
best_model = models['Logistic Regression (Scaled + L2)'][0]
best_preds = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
plt.title('Confusion Matrix: Logistic Regression (Scaled)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()
